## Imports

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.linear_model import Ridge
import shap
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

## Loading Data

Loading fourseam data set, handedness included

In [ ]:
df_ff = pd.read_csv("/Users/suma/Downloads/Baseball_Project/CSV_files/four_seam/df_ff.csv")

In [ ]:
df_ff.shape

## Conference Team Shortcuts

In [ ]:
cusa_teams = [
    "DAL_PAT",   # Dallas Baptist
    "WES_HIL",   # WKU / Western Kentucky
    "KEN_OWL",   # Kennesaw State
    "JAC_GAM",   # Jax State
    "LOU_BUL",   # LA Tech
    "FLO_PAN",   # FIU
    "NMS_AGG",   # NM State
    "LIB_FLA",   # Liberty
    "MTSU_BLU",  # MTSU
    "SAM_BEA",   # Sam Houston
    "UTS_ROA",   # UTSA
    "CHA_FOR",   # Charlotte
    "FAU_OWL",   # FAU
    "RIC_OWL",   # Rice
    "UAB_BLA"    # UAB
]

In [ ]:
sunbelt_teams = [
    "APP_MOU",   # App State
    "ASU_RED",   # Arkansas State
    "COA_CHA",   # Coastal Carolina
    "GEO_EAG",   # Georgia Southern
    "GEO_PAN",   # Georgia State
    "JMU_DUK",   # James Madison
    "LOU_CAJ",   # Louisiana
    "MAR_THU",   # Marshall
    "OLD_MON",   # Old Dominion
    "SAL_JAG",   # South Alabama
    "SOU_GOL",   # Southern Miss
    "TEX_BOB",   # Texas State
    "TRO_TRJ",   # Troy
    "ULM_WAR"    # ULM
]

In [ ]:
american_teams = [
    "ECU_PIR",   # East Carolina
    "HOU_COU",   # Houston
    "WIC_SHO",   # Wichita State
    "UCF_KNI",   # UCF
    "CIN_BEA",   # Cincinnati
    "MT",        # Memphis
    "TUL_GRE",   # Tulane
    "USF_BUL",   # South Florida
    "UTS_ROA",   # UTSA
    "UAB_BLA",   # UAB
    "FAU_OWL",   # Florida Atlantic
    "CHA_FOR",   # Charlotte
    "RIC_OWL"    # Rice
]

In [ ]:
all_three_conference_teams = [
    # C-USA
    "DAL_PAT",   # Dallas Baptist
    "WES_HIL",   # WKU / Western Kentucky
    "KEN_OWL",   # Kennesaw State
    "JAC_GAM",   # Jax State
    "LOU_BUL",   # LA Tech
    "FLO_PAN",   # FIU
    "NMS_AGG",   # NM State
    "LIB_FLA",   # Liberty
    "MTSU_BLU",  # MTSU
    "SAM_BEA",   # Sam Houston

    # Shared C-USA / American
    "UTS_ROA",   # UTSA
    "CHA_FOR",   # Charlotte
    "FAU_OWL",   # FAU
    "RIC_OWL",   # Rice
    "UAB_BLA",   # UAB

    # Sun Belt
    "APP_MOU",   # App State
    "ASU_RED",   # Arkansas State
    "COA_CHA",   # Coastal Carolina
    "GEO_EAG",   # Georgia Southern
    "GEO_PAN",   # Georgia State
    "JMU_DUK",   # James Madison
    "LOU_CAJ",   # Louisiana
    "MAR_THU",   # Marshall
    "OLD_MON",   # Old Dominion
    "SAL_JAG",   # South Alabama
    "SOU_GOL",   # Southern Miss
    "TEX_BOB",   # Texas State
    "TRO_TRJ",   # Troy
    "ULM_WAR",   # ULM

    # American only
    "ECU_PIR",   # East Carolina
    "HOU_COU",   # Houston
    "WIC_SHO",   # Wichita State
    "UCF_KNI",   # UCF
    "CIN_BEA",   # Cincinnati
    "MT",        # Memphis
    "TUL_GRE",   # Tulane
    "USF_BUL"    # South Florida
]

## Filter Conference Matchups

This keeps only rows where both teams involved in the pitch are in combined conference list.

In [ ]:
df_ff_conf = df_ff[
    df_ff["PitcherTeam"].isin(all_three_conference_teams) &
    df_ff["BatterTeam"].isin(all_three_conference_teams)
]

In [ ]:
df_ff_conf

Confirming no outside teams remain

In [ ]:
set(df_ff_conf["PitcherTeam"]).union(df_ff_conf["BatterTeam"]) - set(all_three_conference_teams)

saving data to drive

In [ ]:
df_ff_conf.to_csv(
    "/Users/suma/Downloads/Baseball_Project/CSV_files/four_seam/df_ff_conf.csv",
    index=False
)

In [ ]:
df_ff_conf = pd.read_csv(
    "/Users/suma/Downloads/Baseball_Project/CSV_files/four_seam/df_ff_conf.csv"
)

importing linear regression model results which coach follows

In [ ]:
df_lr = pd.read_excel(
    "/Users/suma/Downloads/Baseball_Project/CSV_files/four_seam/Stuff+ Linear Regressoin.xlsx",
    engine="openpyxl"
)

In [ ]:
df_lr.columns

In [ ]:
df_lr.head()

knowing wwhich features they used

In [ ]:
df_lr["Feature"].unique()

limiting my modeling to the features which they used in the model

In [ ]:
features = [
    'HorzBreak',
    'InducedVertBreak',
    'RelHeight',
    'RelSide',
    'velocity_differential',
    'EffectiveVelo',
    'SpinRate'
]

In [ ]:
X = df_ff_conf[features]
y = df_ff_conf["Target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Model Training and Evaluation

modeling only with filtered features

1. linear regression model

In [ ]:
X = df_ff_conf[features]
y = df_ff_conf["Target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2:", r2_score(y_test, y_pred))

2. ridge regression

In [ ]:
ridge_model = Ridge(alpha=1.0)

ridge_model.fit(X_train, y_train)

y_pred_ridge = ridge_model.predict(X_test)

print("Ridge MAE:", mean_absolute_error(y_test, y_pred_ridge))
print("Ridge RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_ridge)))
print("Ridge R2:", r2_score(y_test, y_pred_ridge))

3. random forest regression

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("RF MAE:", mean_absolute_error(y_test, y_pred_rf))
print("RF RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))
print("RF R2:", r2_score(y_test, y_pred_rf))

4. HistGradientBoosting

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

models = {
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        random_state=42
    ),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.03,
        max_leaf_nodes=31,
        random_state=42
    )
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": np.sqrt(mean_squared_error(y_test, preds)),
        "R2": r2_score(y_test, preds)
    })

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df

In [ ]:
baseline_pred = np.full(len(y_test), y_train.mean())

baseline_results = {
    "Model": "Baseline Mean",
    "MAE": mean_absolute_error(y_test, baseline_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, baseline_pred)),
    "R2": r2_score(y_test, baseline_pred)
}

baseline_results

fine tuning better performed models

In [ ]:
models = {
    "RF tuned": RandomForestRegressor(
        n_estimators=500,
        max_depth=12,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    ),
    "ExtraTrees tuned": ExtraTreesRegressor(
        n_estimators=500,
        max_depth=12,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )
}

In [ ]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": np.sqrt(mean_squared_error(y_test, preds)),
        "R2": r2_score(y_test, preds)
    })

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df

Random Forest:
MAE   0.167054
RMSE  0.254517
R2    0.035767

Extra Trees:
MAE   0.164750
RMSE  0.25526
R2    0.030119

## Delaware SHAP Analysis

importing del_blu data set for shap analysis using linear regression model

In [ ]:
df_del_blu_ff = pd.read_csv(
    "/Users/suma/Downloads/Baseball_Project/CSV_files/four_seam/df_del_blu_ff_cleaned.csv"
)

In [ ]:
df_del_blu_ff.shape

In [ ]:
X_del_blu = df_del_blu_ff[features]

In [ ]:
explainer = shap.LinearExplainer(lr_model, X_train)
shap_values_del_blu = explainer(X_del_blu)

In [ ]:
shap.summary_plot(shap_values_del_blu.values, X_del_blu)
shap.summary_plot(shap_values_del_blu.values, X_del_blu, plot_type="bar")

Pitch level shao

In [ ]:
shap_df = pd.DataFrame(
    shap_values_del_blu.values,
    columns=features,
    index=df_del_blu_ff.index
)

In [ ]:
# Add pitcher name to each pitch-level SHAP row
shap_df["Pitcher"] = df_del_blu_ff["Pitcher"].values

In [ ]:
# Move Pitcher to first column
cols = ["Pitcher"] + [col for col in shap_df.columns if col != "Pitcher"]
shap_df = shap_df[cols]

In [ ]:
shap_df.to_csv(
    "/Users/suma/Downloads/Baseball_Project/CSV_files/four_seam/delaware_lr_shap_pitch_level.csv",
    index=False
)

In [ ]:
shap_df

In [ ]:
# Pitcher-level average SHAP values
pitcher_shap_del = shap_df.groupby("Pitcher").mean()

In [ ]:
# Total average SHAP contribution per pitcher
pitcher_shap_del["raw_shap_sum"] = pitcher_shap_del.sum(axis=1)

In [ ]:
# Sort pitchers by total SHAP impact
pitcher_shap_del = pitcher_shap_del.sort_values("raw_shap_sum", ascending=False)

pitcher_shap_del

normalised scores

In [ ]:
# Mean and standard deviation of pitcher raw SHAP totals
mean_val = pitcher_shap_del["raw_shap_sum"].mean()
std_val = pitcher_shap_del["raw_shap_sum"].std()


In [ ]:
# 1 standard deviation = 15 points
scaling_factor = 15 / std_val

In [ ]:
# Pitcher-friendly normalized score
# lower raw_shap_sum => higher normalized_score
pitcher_shap_del["normalized_score"] = (
    100 - (pitcher_shap_del["raw_shap_sum"] - mean_val) * scaling_factor
)

In [ ]:
pitcher_shap_del

In [ ]:
coach_scores = pd.DataFrame({
    "Pitcher": [
        "Conway, Elias",
        "Shaub, Ethan",
        "Minckler, Matthew",
        "Pollaro, Jake",
        "Callaway, Andrew",
        "Richardson, Jonah",
        "Lewis, Evan",
        "Hartman, Timothy",
        "Marose, Doug",
        "McLaughlin, Ryan",
        "Bryan, Dylan",
        "Moyzan, Ben",
        "Blum, Brady",
    ],
    "coach_score": [
        125, 74, 78, 124, 107, 116, 96, 132, 84, 86, 72, 134, 102
    ]
})

In [ ]:
comparison_table = pitcher_shap_del.reset_index()[["Pitcher", "normalized_score"]].merge(
    coach_scores,
    on="Pitcher",
    how="left"
)

comparison_table

In [ ]:
comparison_table = comparison_table.sort_values(
    "normalized_score",
    ascending=False
)

comparison_table

In [ ]:
comparison_table_no_nan = comparison_table.dropna(subset=["coach_score"]).copy()

In [ ]:
comparison_table_no_nan = comparison_table_no_nan.sort_values(
    "normalized_score",
    ascending=False
)

comparison_table_no_nan

COmaprision table for linear regression model

In [ ]:
comparison_table_no_nan["normalized_score"] = (
    comparison_table_no_nan["normalized_score"].round(0).astype(int)
)

comparison_table_no_nan["coach_score"] = (
    comparison_table_no_nan["coach_score"].round(0).astype(int)
)

comparison_table_no_nan

shap analysis for random forest model

In [ ]:
explainer_rf = shap.TreeExplainer(rf_model)
shap_values_rf_del_blu = explainer_rf(
    X_del_blu,
    approximate=True
)

Create SHAP dataframe:

In [ ]:
shap_rf_df = pd.DataFrame(
    shap_values_rf_del_blu.values,
    columns=features,
    index=df_del_blu_ff.index
)

Add pitcher:

In [ ]:
shap_rf_df["Pitcher"] = df_del_blu_ff["Pitcher"].values

cols = ["Pitcher"] + [col for col in shap_rf_df.columns if col != "Pitcher"]
shap_rf_df = shap_rf_df[cols]

Pitcher-level average SHAP:

In [ ]:
pitcher_shap_rf_del = shap_rf_df.groupby("Pitcher").mean()

Raw SHAP sum:

In [ ]:
pitcher_shap_rf_del["raw_shap_sum"] = pitcher_shap_rf_del.sum(axis=1)

Normalize pitcher-friendly, where lower predicted Target is better:

In [ ]:
mean_val_rf = pitcher_shap_rf_del["raw_shap_sum"].mean()
std_val_rf = pitcher_shap_rf_del["raw_shap_sum"].std()

scaling_factor_rf = 15 / std_val_rf

pitcher_shap_rf_del["normalized_score"] = (
    100 - (pitcher_shap_rf_del["raw_shap_sum"] - mean_val_rf) * scaling_factor_rf
)

pitcher_shap_rf_del = pitcher_shap_rf_del.sort_values(
    "normalized_score",
    ascending=False
)

pitcher_shap_rf_del

Then merge with coach scores:

In [ ]:
rf_comparison_table = pitcher_shap_rf_del.reset_index()[["Pitcher", "normalized_score"]].merge(
    coach_scores,
    on="Pitcher",
    how="left"
)

rf_comparison_table = rf_comparison_table.dropna(subset=["coach_score"]).copy()
rf_comparison_table["coach_score"] = rf_comparison_table["coach_score"].astype(int)

rf_comparison_table

comparision table for random forest model

In [ ]:
rf_comparison_table["normalized_score"] = (
    rf_comparison_table["normalized_score"].round(0).astype(int)
)

rf_comparison_table